<a href="https://colab.research.google.com/github/akashde1998-Alpha/GCN-by-pytorch-/blob/main/GCN_via_pytorchgeometric.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install Required Libraries
This cell installs `torch` and `torch_geometric`, which are essential libraries for building and working with Graph Neural Networks.

In [5]:
!pip install torch
!pip install torch_geometric



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.5 MB/s eta 0:00:00


### Import Necessary Modules
This cell imports various modules from `torch`, `torch.nn.functional`, and `torch_geometric` that will be used for defining the GCN model, handling data, and applying transformations.

In [6]:
import torch
import torch.nn.functional as F

import torch_geometric
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.datasets import Planetoid


### Load the Cora Dataset
This cell loads the Cora dataset, a commonly used benchmark for graph-based machine learning. `NormalizeFeatures()` is applied to normalize the node features, and the first graph in the dataset (`dataset[0]`) is assigned to the `data` variable.

In [7]:
dataset=torch_geometric.datasets.Planetoid(root='/Cora',name='Cora',transform=NormalizeFeatures())
data=dataset[0]

Processing...
Done!


### Inspect Dataset Properties
This cell prints various properties of the loaded Cora dataset, such as an example node's features, the total number of features per node, the number of classes, the total number of nodes, and the total number of edges. This helps in understanding the dataset's structure.

In [8]:
print(data)
print(data.x[36])
print(data.num_features)
print(dataset.num_classes)
print(data.num_nodes)
print(data.num_edges)


Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
tensor([0.0455, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000])
1433
7
2708
10556


### GCN Model Definition and Instantiation
This cell defines a `GCN` model with three convolutional layers. The first layer maps input features to 16 hidden channels, the second layer maps 16 channels to 10 channels, and the third layer maps to the number of output classes. It includes ReLU activation and dropout after the first two layers. An instance of this `GCN` model is then created with `hidden_channels=16` (though the intermediate channels are hardcoded within the class definition), and its architecture is printed.

In [9]:
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
  def __init__(self, hidden_channels):
    super().__init__()
    torch.manual_seed(123456)
    self.conv1=GCNConv(data.num_features,16)
    self.conv2=GCNConv(16, 10)
    self.conv3=GCNConv(10, dataset.num_classes)
  def forward(self, x, edge_index ):
     x=self.conv1(x, edge_index)
     x=x.relu()
     # Dropout: randomly drops features during training.
     x=F.dropout(x,p=0.5, training=self.training)   # This line can be removed if we do not want to use dropout.
     x=self.conv2(x, edge_index)
     x=x.relu()
     x=F.dropout(x,p=0.5, training=self.training)
     x=self.conv3(x, edge_index)
     return x

# Create an instance of the model and then print it
model = GCN(hidden_channels=16)
print(model)

GCN(
  (conv1): GCNConv(1433, 16)
  (conv2): GCNConv(16, 10)
  (conv3): GCNConv(10, 7)
)


### Optimizer and Loss Function Initialization
This cell initializes the `optimizer` (Stochastic Gradient Descent) which will be used to update the model's parameters during training, and the `criterion` (Cross Entropy Loss) which will measure the difference between the model's predictions and the true labels.

In [16]:
optimizer=torch.optim.Adam(model.parameters(), lr=0.01)
criterion=torch.nn.CrossEntropyLoss()

### Training Function Definition
This cell defines the `train` function, which performs a single training step. It sets the model to training mode, clears previous gradients, performs a forward pass, calculates the loss, computes gradients using backpropagation, and updates the model's parameters.

In [17]:
def train():
  model.train()  # Put model in training mode
  optimizer.zero_grad() # Clear old gradients
  out=model(data.x, data.edge_index)# Forward pass
  loss=criterion(out[data.train_mask],data.y[data.train_mask]) # Calculate loss only on training nodes
  loss.backward()  # Backpropagation
  optimizer.step() # Update model parameters
  return loss

### `train()` Function — Example

Suppose for one training node, model produces the logits

$$
[0.2,\;1.5,\;0.7]
$$

and the true class is

$$
y=1.
$$

The training process is:

1. **Training mode**

   ```python
   model.train()
   ```

   Puts the model in training mode.

2. **Clear old gradients**

   ```python
   optimizer.zero_grad()
   ```

   Removes gradients from the previous step.

3. **Forward pass**

   ```python
   out = model(data.x, data.edge_index)
   ```

   The  model takes the graph and node features and produces logits:

   $$
   [0.2,\;1.5,\;0.7]
   $$

4. **Calculate loss**

   ```python
   loss = criterion(out[data.train_mask], data.y[data.train_mask])
   ```

   For this node, the true class is \(1\). Cross-Entropy Loss first obtains the probability of the true class:

#### Convert Logits to Probabilities using Softmax

The `CrossEntropyLoss` criterion internally applies the Softmax function to the raw logits to convert them into probabilities. The Softmax function is defined as:

$$ \text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j} e^{z_j}} $$
So,
   $$
   [0.158,\;0.579,\;0.263]
   $$

   Hence,

   $$
   L=-\log(0.579)\approx0.547.
   $$

5. **Backpropagation**

   ```python
   loss.backward()
   ```

   Calculates the gradients of the loss with respect to the model parameters.

6. **Update parameters**

   ```python
   optimizer.step()
   ```

   Uses the gradients to update the model parameters so that the loss can decrease.

7. **Return loss**

   ```python
   return loss
   ```

   Returns the loss value.


### Test Function Definition

This cell defines the `test` function, which evaluates the model's performance on the test dataset. It sets the model to evaluation mode, performs a forward pass, calculates the predicted labels, and then computes the accuracy by comparing predictions with the true labels for the test set.

In [18]:
def test():
  model.eval()# Put model in evaluation mode
  out=model(data.x, data.edge_index)# Forward pass
  pred=out.argmax(dim=1) # Select class with highest score
  test_correct=(pred[data.test_mask]==data.y[data.test_mask])# Compare predictions with true labels
  test_acc=(int(test_correct.sum())/ int(data.test_mask.sum())) # Calculate accuracy
  return test_acc


### `test()` Function — Example
Suppose the model produces:

out =
```
[
    [0.2, 1.5, 0.7],
    [2.1, 0.3, 0.8],
    [0.1, 0.4, 2.2]
]
```

True classes:

`data.y = [1, 2, 2]`

And the test mask is:

`data.test_mask = [True, True, False]`

1. Get predictions
`pred = out.argmax(dim=1)`

Take the largest value from each row:
```
[0.2, 1.5, 0.7] → class 1
[2.1, 0.3, 0.8] → class 0
[0.1, 0.4, 2.2] → class 2
```
Therefore:

`pred = [1, 0, 2]`

2. Select only test nodes
`pred[data.test_mask]`

Since:

`test_mask = [True, True, False]`

we select the first two predictions:

`pred[test_mask] = [1, 0]`

Similarly:

`data.y[data.test_mask]`

gives the true labels of the test nodes:

`data.y[test_mask] = [1, 2]`

3. Compare predictions with true labels
`test_correct = (pred[data.test_mask] == data.y[data.test_mask])`

So:

`pred[test_mask]`   = `[1, 0]`
true labels       = `[1, 2]`

                  ↓

                    [1 == 1, 0 == 2]

                  ↓

                    [True, False]

There is one correct prediction and one incorrect prediction.

4. Count correct predictions
`test_correct.sum()`

Since:

`[True, False]`

is treated as:

`[1, 0]`

we get:

`test_correct.sum() = 1`

5. Count total test nodes
`data.test_mask.sum()`

There are two True values:

`[True, True, False]`
     ↓      ↓

       2 test nodes

Therefore:

`data.test_mask.sum() = 2`

6. Calculate accuracy
`test_acc = 1 / 2`

Therefore:

`test_acc = 0.5 = 50%`

In [19]:
for epoch in range(1, 200):

    loss = train()

    print(f'Epoch: {epoch:03d}, ' f'Loss: {loss:.4f}')
    #print(f"Epoch: {epoch}, Loss: {loss}")
    #print("Epoch:", epoch, "Loss:", loss)

Epoch: 001, Loss: 1.9448
Epoch: 002, Loss: 1.9432
Epoch: 003, Loss: 1.9418
Epoch: 004, Loss: 1.9365
Epoch: 005, Loss: 1.9306
Epoch: 006, Loss: 1.9223
Epoch: 007, Loss: 1.9186
Epoch: 008, Loss: 1.9037
Epoch: 009, Loss: 1.9030
Epoch: 010, Loss: 1.8921
Epoch: 011, Loss: 1.8712
Epoch: 012, Loss: 1.8596
Epoch: 013, Loss: 1.8528
Epoch: 014, Loss: 1.8486
Epoch: 015, Loss: 1.8261
Epoch: 016, Loss: 1.8005
Epoch: 017, Loss: 1.7900
Epoch: 018, Loss: 1.7827
Epoch: 019, Loss: 1.7551
Epoch: 020, Loss: 1.7539
Epoch: 021, Loss: 1.7274
Epoch: 022, Loss: 1.7082
Epoch: 023, Loss: 1.6911
Epoch: 024, Loss: 1.6715
Epoch: 025, Loss: 1.6086
Epoch: 026, Loss: 1.6296
Epoch: 027, Loss: 1.5836
Epoch: 028, Loss: 1.5744
Epoch: 029, Loss: 1.5666
Epoch: 030, Loss: 1.5102
Epoch: 031, Loss: 1.5134
Epoch: 032, Loss: 1.4733
Epoch: 033, Loss: 1.4464
Epoch: 034, Loss: 1.4131
Epoch: 035, Loss: 1.4074
Epoch: 036, Loss: 1.3549
Epoch: 037, Loss: 1.3306
Epoch: 038, Loss: 1.2838
Epoch: 039, Loss: 1.2630
Epoch: 040, Loss: 1.2421


### Evaluate Model Performance

This cell evaluates the trained model on the test dataset and prints the final test accuracy. The `test()` function calculates how well the model predicts the correct labels for the nodes in the test set.

In [21]:
test_acc = test()
print("test_acc:", test_acc)

test_acc: 0.73
